In [10]:
import numpy as np
import pandas as pd
pd.options.display.float_format = '{:,.3f}'.format

In [11]:
df = pd.read_parquet("mortgage_data.parquet", engine="pyarrow")

In [12]:
# Create A/B groups randomly
df["group"] = np.random.choice(["A", "B"], size=len(df))

# A will be the control group
df["rate_diff_A"] = df["rate_diff"]

# B will have a rate improvement of 0.125% (12.5 bps)
df["rate_diff_B"] = df["rate_diff"] - 0.125

In [13]:
# re use compute_lock_probability from pricing Elasticity notebook
def compute_lock_prob(df, new_rate_diff):

    # borrowre starts slightly lower than 50% tendency to lock
    logit = -0.50 

    # if new rate_diff is negative, quote is better than market, negative becomes positive
    # lock probablity goes up; better pricing relative to market increases lock probability
    logit += -5.0 * new_rate_diff

    # purchase helps most, refinance a little, cashout hurts
    logit += np.where(df["purpose"] == "purchase", 0.45, 0)
    logit += np.where(df["purpose"] == "refinance", 0.10, 0)
    logit += np.where(df["purpose"] == "cash_out", -0.25, 0)

    # primary helps, second home hurts a little, investment hurts most
    logit += np.where(df["occupancy"] == "primary", 0.15, 0)
    logit += np.where(df["occupancy"] == "second_home", -0.10, 0)
    logit += np.where(df["occupancy"] == "investment", -0.30, 0)

    # higher fico helps but moderately
    logit += 0.003 * (df["fico"] - 700)

    # higher ltv hurts, lower ltv helps
    logit += -0.015 * (df["ltv"] - 80)

    # large loans reduce lock probability
    logit += -0.000001 * (df["loan_amount"] - 450000)

    # market volatility, low volatility helps, high hurts
    logit += np.where(df["market_volatility"] == "low", 0.20, 0)
    logit += np.where(df["market_volatility"] == "medium", 0.00, 0)
    logit += np.where(df["market_volatility"] == "high", -0.35, 0)
    
    # if loan is refinance, pricig isbetter than market, give boost
    logit += np.where((df["purpose"] == "refinance") & (new_rate_diff < 0), 0.35, 0)
    
    # converts logit into a probability between 0 and 1
    lock_prob = 1 / (1 + np.exp(-logit))

    # prevents extreme values
    lock_prob = np.clip(lock_prob, 0.02, 0.98)
    
    return lock_prob

In [14]:
# compute probabilities for both groups
df["prob_A"] = compute_lock_prob(df, df["rate_diff_A"])
df["prob_B"] = compute_lock_prob(df, df["rate_diff_B"])

In [15]:
# simulate actual outcomes
df["locked_A"] = np.random.binomial(1, df["prob_A"])
df["locked_B"] = np.random.binomial(1, df["prob_B"])

In [16]:
df["locked"] = np.where(df["group"] == "A", df["locked_A"], df["locked_B"])

In [17]:
margin_A = 125
margin_B = 112.5

df['revenue'] = np.where(
    df["group"] == "A",
    df["loan_amount"] * (margin_A / 10000) * df["locked_A"],
    df["loan_amount"] * (margin_B / 10000) * df["locked_B"]
)

In [18]:
ab_results = df.groupby("group").agg({
    "locked": "mean",
    "loan_amount": "count",
    "revenue": "sum"
}).rename(columns={
    "locked": "conversion_rate",
    "loan_amount": "num_loans"
})

print(ab_results)

       conversion_rate  num_loans           revenue
group                                              
A                0.506     749942 2,082,708,206.013
B                0.644     750058 2,405,183,826.409


In [19]:
from scipy.stats import ttest_ind

ttest_ind(
    df[df["group"] == "A"]["locked"],
    df[df["group"] == "B"]["locked"]
)

TtestResult(statistic=-172.16877505696814, pvalue=0.0, df=1499998.0)

In [20]:
df.groupby("group")["locked"].mean()

group
A   0.506
B   0.644
Name: locked, dtype: float64

In [21]:
conversion_summary = df.groupby("group")["locked"].mean()
lift = conversion_summary["B"] - conversion_summary["A"]
print(conversion_summary)
print("Absolute lift:", round(lift, 4))

group
A   0.506
B   0.644
Name: locked, dtype: float64
Absolute lift: 0.1376


In [22]:
revenue_summary = df.groupby("group")["revenue"].sum()
print(revenue_summary)
print("Revenue difference:", round(revenue_summary["B"] - revenue_summary["A"], 0))

group
A   2,082,708,206.013
B   2,405,183,826.409
Name: revenue, dtype: float64
Revenue difference: 322475620.0


In [23]:
ab_summary = df.groupby("group").agg(
    conversion_rate=("locked", "mean"),
    loans=("locked", "size"),
    total_revenue=("revenue", "sum"),
    avg_revenue_per_loan=("revenue", "mean")
)

ab_summary["conversion_rate"] = ab_summary["conversion_rate"].round(4)
ab_summary["total_revenue"] = ab_summary["total_revenue"].round(0)
ab_summary["avg_revenue_per_loan"] = ab_summary["avg_revenue_per_loan"].round(2)

print(ab_summary)

       conversion_rate   loans     total_revenue  avg_revenue_per_loan
group                                                                 
A                0.506  749942 2,082,708,206.000             2,777.160
B                0.644  750058 2,405,183,826.000             3,206.660


In [24]:
conversion = df.groupby("group")["locked"].mean()

lift = conversion["B"] - conversion["A"]
lift_pct = lift / conversion["A"]

print("Lift:", round(lift, 4))
print("Lift %:", round(lift_pct, 4))

Lift: 0.1376
Lift %: 0.272


In [25]:
revenue = df.groupby("group")["revenue"].sum()

rev_lift = revenue["B"] - revenue["A"]
rev_lift_pct = rev_lift / revenue["A"]

print("Revenue Lift:", round(rev_lift, 0))
print("Revenue Lift %:", round(rev_lift_pct, 4))

Revenue Lift: 322475620.0
Revenue Lift %: 0.1548


In [26]:
avg_rev = df.groupby("group")["revenue"].mean()
print(avg_rev)

group
A   2,777.159
B   3,206.664
Name: revenue, dtype: float64


In [27]:
ab_summary = df.groupby("group").agg(
    conversion_rate=("locked", "mean"),
    total_loans=("locked", "size"),
    total_revenue=("revenue", "sum"),
    avg_revenue_per_loan=("revenue", "mean")
)

# Add lift vs A
ab_summary["conversion_lift"] = ab_summary["conversion_rate"] - ab_summary.loc["A", "conversion_rate"]
ab_summary["revenue_lift"] = ab_summary["total_revenue"] - ab_summary.loc["A", "total_revenue"]

print(ab_summary)

       conversion_rate  total_loans     total_revenue  avg_revenue_per_loan  \
group                                                                         
A                0.506       749942 2,082,708,206.013             2,777.159   
B                0.644       750058 2,405,183,826.409             3,206.664   

       conversion_lift    revenue_lift  
group                                   
A                0.000           0.000  
B                0.138 322,475,620.396  
